In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression

In [ ]:
# Local replacement for Jake's cells 1-5. His notebook built the target
# series from two internal lab exports (PL-FTH-INF_cleaned.csv and
# PL-FTH-HW_cleaned.csv) that were not shared. FoothillsInfluent.csv is
# the output of that step, so it is loaded directly here.
from pathlib import Path
DATA = Path("../data")
usgs = pd.read_csv(DATA / "USGS_South_Platte.csv")
dwr = pd.read_csv(DATA / "SouthPlatteTelemetry.csv")
precip = pd.read_csv(DATA / "USC00058022.csv")
sntl = pd.read_csv(DATA / "HoosierPass.csv")
fth = pd.read_csv(DATA / "FoothillsInfluent.csv")
fth['DATE'] = pd.to_datetime(fth['DATE'])
combined_fth = (fth.set_index('DATE')
                   .rename(columns={'TOC_mg_L': 'TOC', 'Alk_mg_L': 'Alk'})
                   [['TOC']].sort_index())
combined_fth


In [ ]:
# Set USGS index to datetime index, remove timezone
usgs['DATE'] = pd.to_datetime(usgs['Date']).dt.tz_localize(None)
usgs = usgs.drop(columns = 'Date')
usgs.set_index('DATE', inplace=True)

# Lag the USGS dataframe by 2 days
usgs_lagged = usgs.shift(2, freq='D')

# join dfs based on shared index
combined_df = combined_fth.join(usgs_lagged, how="left")

In [ ]:
# Format snotel data, lag by 2 days, and merge
sntl['DATE'] = pd.to_datetime(sntl['DATE'])
sntl.set_index('DATE', inplace=True)
sntl['SWE'] = pd.to_numeric(sntl['SWE'], errors="coerce")
sntl_lagged = sntl.shift(2, freq='D')
combined_df = combined_df.join(sntl_lagged, how="left")

In [ ]:
# Format DWR flow data, lag by 2 days, and merge
dwr['DATE'] = pd.to_datetime(dwr['Date'])
dwr = dwr.drop(columns = 'Date')
dwr.set_index('DATE', inplace=True)
dwr_lagged = dwr.shift(2, freq='D')
combined_df = combined_df.join(dwr_lagged, how="left")

In [ ]:
# Format NOAA precip data, lag by 4 days, and merge 
# always 2 days behind present day, so need longer lag for operational constraints.
precip['DATE'] = pd.to_datetime(precip['DATE'])
precip.set_index('DATE', inplace=True)
precip_lagged = precip.shift(4, freq='D')
combined_df = combined_df.join(precip_lagged, how="left")

In [ ]:
# Add a month column to use as a predictor
combined_df['Month'] = combined_df.index.month

# Create the angle in radians (2 * pi * month / 12)
# We subtract 1 so January starts at 0
month_radians = 2 * np.pi * (combined_df['Month'] - 1) / 12

# Generate Sine and Cosine components
combined_df['month_sin'] = np.sin(month_radians)
combined_df['month_cos'] = np.cos(month_radians)

# Drop rows where TOC is null
combined_df_TOC = combined_df.dropna(subset=['TOC']).copy()

In [ ]:
# Adding more engineered features for prediction 
# Is the river rising or falling?
combined_df_TOC['flow_delta'] = combined_df_TOC['Flow_CFS'].diff() 

# 7-day rolling average (the 'Baseline' state of the watershed)
combined_df_TOC['flow_7day_avg'] = combined_df_TOC['Flow_CFS'].rolling(window=7).mean()
combined_df_TOC['gage_ht_7day'] = combined_df_TOC['GageHeight_ft'].rolling(window=7).mean()


# Create a turbidity and gage height 'Trend' feature
combined_df_TOC['turb_3day'] = combined_df_TOC['Turbidity_Median'].rolling(window=3).mean()
combined_df_TOC['gage_ht_3day'] = combined_df_TOC['GageHeight_ft'].rolling(window=3).mean()

# Create a DOY feature
day_of_year = combined_df.index.dayofyear
combined_df_TOC['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365)
combined_df_TOC['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365)

# turbidity "loading" feature
combined_df_TOC['turb_flow'] = combined_df_TOC['turb_3day'] * combined_df_TOC['flow_7day_avg']

# Experimental precip features
combined_df_TOC['precip_7day'] = combined_df_TOC['PRCP'].rolling(window=7).mean()
combined_df_TOC['precip_3day'] = combined_df_TOC['PRCP'].rolling(window=3).mean()
combined_df_TOC['precip_daily_dwr'] = combined_df_TOC['Precip'].diff() # don't use this as a predictor. Data is really dirty. NOAA preferred. 
combined_df_TOC['swe_7day'] = combined_df_TOC['SWE'].rolling(window=7).mean()
combined_df_TOC['swe_3day'] = combined_df_TOC['SWE'].rolling(window=3).mean()

#Experimental specific_cond features
combined_df_TOC['cond_7day'] = combined_df_TOC['Specific_Cond_Mean'].rolling(window=7).mean()
combined_df_TOC['cond_3day'] = combined_df_TOC['Specific_Cond_Mean'].rolling(window=3).mean()
combined_df_TOC['turb_cond'] = combined_df_TOC['Specific_Cond_Mean'] * combined_df_TOC['turb_3day']
combined_df_TOC['turb/cond'] = combined_df_TOC['Specific_Cond_Mean'] / combined_df_TOC['turb_3day']

In [ ]:
# Calculate the correlation matrix
corr_matrix = combined_df_TOC.corr(numeric_only=True, method='spearman')

# Keep only correlations with TOC
toc_corr = corr_matrix[['TOC']].sort_values('TOC', ascending=False)

plt.figure(figsize=(8, 10))
sns.heatmap(
    toc_corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    center=0
)

plt.title('Spearman Correlation with Turb')
plt.tight_layout()
plt.savefig("figures/TOCCorrelationMatrix_spearman.png")
plt.show()

In [ ]:
# Building a quick linear regression with turb_flow to compare to the final model
# 1. Define the model features
feature = ['turb_flow']
target = 'TOC' 

# Select only features + the target, then drop rows with NaNs
df_linear_model = combined_df_TOC[feature + [target]].dropna()

# Set X and y
X = df_linear_model[feature]
y = df_linear_model[target]

# 2. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, shuffle = False
)

# 3. Initialize the model
model = LinearRegression()

# 4. Train the model using the training data
model.fit(X_train, y_train)

# 5. Make predictions on the train and test data
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# 6. Extract learned mathematical properties
print(f"Slope (Coefficient): {model.coef_[0]:.4f}")
print(f"Y-Intercept: {model.intercept_:.4f}")

# 7. Evaluate model performance
print(f"Train RMSE: {root_mean_squared_error(y_train, y_train_pred):.4f}")
print(f"Training R-squared Score: {r2_score(y_train, y_train_pred):.4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, y_pred):.4f}")
print(f"Test R-squared Score: {r2_score(y_test, y_pred):.4f}")

# 8. Plot the model
plt.figure(figsize=(12, 6))

# Plot actual values
plt.scatter(y_test.index, y_test.values, label='Measured Foothills Influent Turb', 
         color='blue', marker='o', alpha=0.8)

# Plot predicted values
plt.plot(y_test.index, y_pred, label='Soft Sensor Predicted TOC (Linear regression)', 
         color='orange', linestyle='--', linewidth=2)

plt.title(f"TOC Soft Sensor Performance")
plt.xlabel("Date")
plt.ylabel("TOC (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.show()

In [ ]:
# Assign predictors and target columns
features = ['Specific_Cond_Mean', 'month_cos', 'month_sin', 'turb_flow', 'turb_3day', 'precip_7day', 'swe_7day',
           'turb/cond', 'Turbidity_Median', 'Turbidity_Max']
target = 'TOC' 

# Select only features + the target, then drop rows with NaNs
df_model = combined_df_TOC[features + [target]].dropna()

# Set X and y
X = df_model[features]
y = df_model[target]

In [ ]:
len(df_model)

In [ ]:
df_model.tail()

In [ ]:
print("Starting Grid Search...")

# 50/50 Split
X_train_cv, X_test, y_train_cv, y_test = train_test_split(
    X, y, test_size=0.5, shuffle=False
)

# Create weights
weights = np.where(y_train_cv > 3.0, 1.5, 1.0)

# Setup the Grid Search
# We focus on depth and leaf size to pull that biased prediction line down
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [5, 10, 20],
    'max_features': [1.0, 'sqrt'] # 1.0 uses all features, sqrt uses a subset
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

# Fit Grid Search on the Training set
grid_search.fit(X_train_cv, y_train_cv, sample_weight=weights)

# Extracting mean, std dev, and last fold R2 for the best model
best_index = grid_search.best_index_
mean_cv_r2 = grid_search.cv_results_['mean_test_score'][best_index]
std_cv_r2 = grid_search.cv_results_['std_test_score'][best_index]

# Calculate the index of the last split
last_split_index = grid_search.n_splits_ - 1

# Extract the score for the last fold at the best_index
last_fold_r2 = grid_search.cv_results_[f'split{last_split_index}_test_score'][best_index]

# Print the results
print(f"\nCross-Validation Results")
print(f"Best Mean R2:  {mean_cv_r2:.4f}")
print(f"R2 Std Dev:    {std_cv_r2:.4f}")
print(f"Last Fold R2:  {last_fold_r2:.4f}")

# Extract and Retrain (Final Fit is done automatically by GridSearchCV if refit=True)
best_rf = grid_search.best_estimator_

# Final Evaluation on 40% Test Set
final_preds = best_rf.predict(X_test)

print(f"\nTest MAPE: {mean_absolute_percentage_error(y_test, final_preds):.4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, final_preds):.4f}")
print(f"Test R2: {r2_score(y_test, final_preds):.4f}")

In [ ]:
# Look at training performance with best model on outliers
y_preds = best_rf.predict(X_train_cv)
# Calculate metrics for the plot title
train_rmse = root_mean_squared_error(y_train_cv, y_preds)

# Create the Comparison Plot
plt.figure(figsize=(12, 6))

# Plot actual values
plt.plot(y_train_cv.index, y_train_cv.values, label='Measured Foothills Influent TOC', 
         color='blue', marker='o', markersize=4, alpha=0.8)

# Plot predicted values
plt.plot(y_train_cv.index, y_preds, label='Soft Sensor Predicted TOC (Optimized RF)', 
         color='orange', linestyle='--', linewidth=2)

plt.title(f"TOC Soft Sensor Performance  (Train RMSE: {train_rmse:.2} mg/L)")
plt.xlabel("Date")
plt.ylabel("TOC (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.savefig("figures/TOCPredictionComparison_Training.png")
plt.show()

In [ ]:
# Calculate metrics for the plot title
final_rmse = root_mean_squared_error(y_test, final_preds)

# Create the Comparison Plot
plt.figure(figsize=(12, 6))

# Plot actual values
plt.plot(y_test.index, y_test.values, label='Measured Foothills Influent TOC', 
         color='blue', marker='o', markersize=4, alpha=0.8)

# Plot predicted values
plt.plot(y_test.index, final_preds, label='Soft Sensor Predicted TOC (Optimized RF)', 
         color='orange', linestyle='--', linewidth=2)

plt.title(f"TOC Soft Sensor Performance  (Test RMSE: {final_rmse:.2} mg/L)")
plt.xlabel("Date")
plt.ylabel("TOC (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.savefig("figures/TOCPredictionComparison.png")
plt.show()

In [ ]:
# Extract importance and pair with feature names
importance = pd.Series(best_rf.feature_importances_, index=X.columns)

# Sort and plot
importance.sort_values().plot(kind='barh', color='skyblue')
plt.title("Random Forest Feature Importance (MDI)")
plt.tight_layout()
plt.savefig("figures/TOCFeatureImportance.png")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    best_rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2, scoring='neg_root_mean_squared_error'
)

sorted_importances_idx = result.importances_mean.argsort()
importances = pd.DataFrame(
    result.importances[sorted_importances_idx].T,
    columns=X.columns[sorted_importances_idx],
)
ax = importances.plot.box(vert=False, whis=10)
ax.set_title("Permutation Importances (test set)")
ax.axvline(x=0, color="k", linestyle="--")
ax.set_xlabel("Decrease in RMSE score")
ax.figure.tight_layout()
plt.savefig("figures/TOCPermutationImportance.png")
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
# Trying normal train/validation/test split to see if it improves validation score

print("Starting Fixed Split Training...")

# 1. Manual Chronological Split (45/10/45)
n_samples = len(X)
train_end = int(n_samples * 0.45)
val_end = int(n_samples * 0.55)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

# 2. Weights for Training (only applied to X_train)
weights = np.where(y_train > 6.0, 1.5, 1.0)

# 3. Manual Grid Search (Since we aren't using GridSearchCV's CV)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [5, 10, 20],
    'max_features': [1.0, 'sqrt']
}

best_val_r2 = -np.inf
best_model = None

# Iterate through hyperparameters
for n_est in param_grid['n_estimators']:
    for depth in param_grid['max_depth']:
        for leaf in param_grid['min_samples_leaf']:
            for feat in param_grid['max_features']:
                
                model = RandomForestRegressor(
                    n_estimators=n_est,
                    max_depth=depth,
                    min_samples_leaf=leaf,
                    max_features=feat,
                    random_state=42
                )
                
                # Fit on Train
                model.fit(X_train, y_train, sample_weight=weights)
                
                # Evaluate on Validation
                val_preds = model.predict(X_val)
                val_r2 = r2_score(y_val, val_preds)
                
                if val_r2 > best_val_r2:
                    best_val_r2 = val_r2
                    best_model = model
                    best_params = {'n_estimators': n_est, 'max_depth': depth, 
                                   'min_samples_leaf': leaf, 'max_features': feat}

# 4. Final Evaluation on Test Set
final_preds = best_model.predict(X_test)

print(f"\nBest Params based on Validation: {best_params}")
print(f"Validation R2: {best_val_r2:.4f}")

print(f"\nFinal Test Performance (Last 45% of data)")
print(f"Test MAPE: {mean_absolute_percentage_error(y_test, final_preds):.4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, final_preds):.4f}")
print(f"Test R2:   {r2_score(y_test, final_preds):.4f}")

In [ ]:
# Calculate metrics for the plot title
final_rmse = root_mean_squared_error(y_test, final_preds)

# Create the Comparison Plot
plt.figure(figsize=(12, 6))

# Plot actual values
plt.plot(y_test.index, y_test.values, label='Measured Foothills Influent TOC', 
         color='blue', marker='o', markersize=4, alpha=0.8)

# Plot predicted values
plt.plot(y_test.index, final_preds, label='Soft Sensor Predicted TOC (Optimized RF)', 
         color='orange', linestyle='--', linewidth=2)

plt.title(f"TOC Soft Sensor Performance  (Test RMSE: {final_rmse:.2} mg/L)")
plt.xlabel("Date")
plt.ylabel("TOC (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.show()

In [ ]:
from catboost import CatBoostRegressor

# 50/50 chronological split
X_train_cv, X_test, y_train_cv, y_test = train_test_split(
    X,
    y,
    test_size=0.5,
    shuffle=False
)

# Emphasize high values
weights = np.where(y_train_cv > 4.0, 1.5, 1.0)

# CatBoost baseline
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function='RMSE',
    random_state=42,
    verbose=100
)

cat_model.fit(
    X_train_cv,
    y_train_cv,
    sample_weight=weights
)

# Predictions
final_preds = cat_model.predict(X_test)

print(f"\nTest MAPE: {mean_absolute_percentage_error(y_test, final_preds):.4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, final_preds):.4f}")
print(f"Test R2:   {r2_score(y_test, final_preds):.4f}")


In [ ]:
importance = pd.Series(
    cat_model.feature_importances_,
    index=X_train_cv.columns
).sort_values(ascending=False)

print(importance)


In [ ]:
# Calculate metrics for the plot title
final_rmse = root_mean_squared_error(y_test, final_preds)

# Create the Comparison Plot
plt.figure(figsize=(12, 6))

# Plot actual values
plt.plot(y_test.index, y_test.values, label='Measured Foothills Influent TOC', 
         color='blue', marker='o', markersize=4, alpha=0.8)

# Plot predicted values
plt.plot(y_test.index, final_preds, label='Soft Sensor Predicted TOC (Base CatBoost)', 
         color='orange', linestyle='--', linewidth=2)

plt.title(f"TOC Soft Sensor Performance  (Test RMSE: {final_rmse:.2} mg/L)")
plt.xlabel("Date")
plt.ylabel("TOC (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.savefig("figures/TOCPredictionComparison_CatBoost.png")
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(cat_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
# Next steps are to add in new sensor data (sonde?) and see if we can remove redunant features/simplify the model